In [ ]:
import os, sys
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import ast
import requests
from bs4 import BeautifulSoup
import json
from typing import List
sys.path.append(os.path.abspath("..")) 
sys.path.append('<PROJECT_ROOT_D3>/')
from Funcs.Utility import *
from tqdm import tqdm
from glob import glob
from time import sleep

from selenium import webdriver as webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options
from selenium.common.exceptions import InvalidSessionIdException
from selenium.webdriver import FirefoxOptions
from selenium.webdriver.common.keys import Keys
import warnings
warnings.filterwarnings("ignore")

# EDA for the KeyEvent Data

In [ ]:
uids = os.listdir(PATH_SENSOR)
df_test = pd.read_csv(os.path.join(PATH_SENSOR, uids[0], 'KeyEvent.csv'))

In [ ]:
df_test

In [ ]:
import os
import pandas as pd
from glob import glob

# Part 1: Get the list of uncategorized apps
UNCATEGORIZED_APPS_PATH = os.path.join(PATH_DATA, "uncategorized_apps.csv")
if os.path.exists(UNCATEGORIZED_APPS_PATH):
    os.remove(UNCATEGORIZED_APPS_PATH)

# Make Entire app List from the data
app_list = set()

for uid in uids:
    # Read all APP_USAGE_EVENT.csv files for each user ID
    app_usage_paths = sorted(glob(os.path.join(PATH_SENSOR, uid, 'KeyEvent.csv')))
    dfs = [pd.read_csv(path, index_col=False, header=0) for path in app_usage_paths]
    
    if dfs:  # Check if dfs list is not empty
        df = pd.concat(dfs, ignore_index=True)
        app_list.update(df['packageName'].unique().tolist())

# Convert set to sorted list
app_list = sorted(app_list)
print("# Total Apps:", len(app_list))

# Create DataFrame similar to the previously shown format and save to CSV
df = pd.DataFrame(app_list, columns=['packageName'])
df.to_csv(UNCATEGORIZED_APPS_PATH, index=False)

print(df.head(), "\n", df.columns, "\n", df.shape, "\n")

In [ ]:
df_test = pd.read_csv(UNCATEGORIZED_APPS_PATH)

In [ ]:
# Part 2: Categorize the apps, first stage
print( "<"*20 + " First Extraction Began " + ">"*20 )
sleep(1)

if os.path.exists(os.path.join(PATH_DATA, "app_category_1.csv")):
    os.remove(os.path.join(PATH_DATA, "app_category_1.csv"))

def extractAppCategory1()->pd.DataFrame:
    search_platform = "https://apkcombo.com"
    opts = FirefoxOptions()
    opts.add_argument("--headless")
    browser = webdriver.Firefox(options=opts)

    df = []
    app_list = pd.read_csv(os.path.join(PATH_DATA, "uncategorized_apps.csv"))['packageName'].tolist()
    for packageName in tqdm(app_list):
        packageName = packageName.lower()
        browser.get(search_platform)

        row = [packageName, None, None, search_platform]

        search = browser.find_element(By.NAME, "q")
        search.send_keys(row[0])
        browser.find_element(By.CLASS_NAME, "button-search").click()
        try:
            search_soup = BeautifulSoup(browser.page_source, 'html.parser')
            
            category_p_tag = search_soup.find('div', text=lambda text: text and 'Category' in text)

            if category_p_tag:
                category = category_p_tag.find_next_sibling("div").text
                row[2] = category.upper()

            title_tag = search_soup.find('div', class_='app_name').findChild('h1').findChild('a').text
            row[1] = title_tag.upper()
        except Exception as e:
            print(f"Error: {e}")
        print(row)
        df.append(row)
    browser.close()
    return pd.DataFrame(df, columns=['packageName', 'appName', 'category', 'source'])

df = extractAppCategory1()
df.drop(columns=[col for col in df.columns if 'Unnamed' in col], inplace=True)
df.to_csv(os.path.join(PATH_DATA, "app_category_1.csv"), index=False)
print(df.isna().sum())

In [ ]:
# ### Clean Version #########

# import os
# import pandas as pd
# import json
# from tqdm import tqdm
# from selenium import webdriver
# from selenium.webdriver.common.by import By
# from selenium.webdriver.firefox.options import Options as FirefoxOptions
# from selenium.webdriver.common.desired_capabilities import DesiredCapabilities
# from selenium.webdriver.common.action_chains import ActionChains
# from selenium.webdriver.support.ui import WebDriverWait
# from selenium.webdriver.support import expected_conditions as EC
# from bs4 import BeautifulSoup
# from time import sleep

# # Part 3: Categorize the apps, second stage
# print("<"*20 + " Second Extraction Began " + ">"*20)
# sleep(1)

# def extractAppCategory2(apps: pd.DataFrame) -> pd.DataFrame:
#     dics = []

#     # Set up browser options
#     opts = FirefoxOptions()
#     opts.add_argument("--headless")  # Remove this line for debugging in non-headless mode
    
#     # Set up capabilities to force English content
#     capabilities = DesiredCapabilities.FIREFOX
#     capabilities["marionette"] = True
#     capabilities["acceptInsecureCerts"] = True
#     opts.set_preference("intl.accept_languages", "en-US, en")

#     # Launch the browser with specified capabilities
#     browser = webdriver.Firefox(options=opts, capabilities=capabilities)
#     actions = ActionChains(browser)

#     for idx, row in tqdm(apps.iterrows(), total=apps.shape[0]):
#         package_id = row['packageName']
#         if pd.notna(row['category']):
#             continue  # Skip if the category is already available

#         row_data = [package_id, None, None, "https://play.google.com"]
#         search_platform = f"https://www.google.com/search?q={package_id}+site:play.google.com"

#         try:
#             # Search for the app on Google
#             browser.get(search_platform)
#             sleep(3)  # Wait for search results to load

#             # Extract the first Google Play link
#             anchor_tags = browser.find_elements(By.TAG_NAME, "a")
#             links = [tag.get_attribute("href") for tag in anchor_tags if tag.get_attribute("href") is not None]
#             links = [link for link in links if "https://play.google.com/store/apps/details" in link]

#             # Debugging: Print out the links being found
#             print("Extracted Links:", links)

#             if links:
#                 # Use the first valid link directly
#                 target_link = links[0]

#                 # Ensure it has the correct base URL and add `hl=en` if needed
#                 if "hl=" not in target_link:
#                     if "?" in target_link:
#                         clean_url = target_link + "&hl=en"
#                     else:
#                         clean_url = target_link + "?hl=en"
#                 else:
#                     clean_url = target_link

#                 print("Navigating to Clean URL:", clean_url)  # Debugging print
#                 browser.get(clean_url)
#                 sleep(5)  # Wait for the Play Store page to load

#                 # Explicit wait for the span containing the app name
#                 try:
#                     WebDriverWait(browser, 30).until(
#                         EC.presence_of_element_located((By.XPATH, "//h1/span[@itemprop='name']"))
#                     )
#                 except Exception as e:
#                     print(f"Explicit wait failed for app name: {e}")

#                 # Scroll down to ensure dynamic content is fully loaded
#                 browser.execute_script("window.scrollTo(0, document.body.scrollHeight);")
#                 sleep(3)  # Wait for the page to fully load after scrolling

#                 # Parse the page with BeautifulSoup
#                 soup = BeautifulSoup(browser.page_source, 'html.parser')

#                 # Extract app name using itemprop attribute
#                 app_name_tag = soup.find('span', {'itemprop': 'name'})
#                 if app_name_tag:
#                     row_data[1] = app_name_tag.text.strip()
#                     print(f"App Name Extracted: {row_data[1]}")
#                 else:
#                     # Fallback method to find the app name in case `itemprop="name"` does not exist
#                     app_name_tag_alt = soup.find('title', {'id': 'main-title'})  # Try finding another tag as an alternative
#                     if app_name_tag_alt:
#                         row_data[1] = app_name_tag_alt.text.strip()
#                         print(f"App Name Extracted (Fallback): {row_data[1]}")

#                 # Extract category using different possible methods
#                 category_text = None

#                 # Method 1: Search for the JSON-LD script with application/ld+json
#                 json_ld_script = soup.find('script', {'type': 'application/ld+json'})
#                 if json_ld_script:
#                     try:
#                         json_data = json.loads(json_ld_script.string)
#                         if "applicationCategory" in json_data:
#                             category_text = json_data["applicationCategory"].strip().upper()
#                     except json.JSONDecodeError as e:
#                         print(f"JSON parsing error: {e}")

#                 # Method 2: Search for a div with itemprop="genre"
#                 if not category_text:
#                     genre_div = soup.find('div', {'itemprop': 'genre'})
#                     if genre_div:
#                         # Try extracting from <span> first
#                         genre_span = genre_div.find('span', {'aria-hidden': 'true'})
#                         if genre_span:
#                             category_text = genre_span.text.strip().upper()
#                         # Try extracting from <a> tag if <span> is not found
#                         if not category_text:
#                             genre_a = genre_div.find('a', {'aria-label': True})
#                             if genre_a:
#                                 category_text = genre_a['aria-label'].strip().upper()

#                 # Method 3: Search for category within a known structure
#                 if not category_text:
#                     category_a_tags = soup.find_all('a', href=True)
#                     for a_tag in category_a_tags:
#                         if "/store/apps/category/" in a_tag['href']:
#                             category_text = a_tag.text.strip().upper()
#                             break

#                 # Method 4: Search by potential keywords for a fallback approach
#                 if not category_text:
#                     potential_spans = soup.find_all('span')
#                     for span in potential_spans:
#                         if 'Category' in span.text:
#                             next_tag = span.find_next()
#                             if next_tag:
#                                 category_text = next_tag.text.strip().upper()
#                                 break

#                 # Update the row_data if category_text is found
#                 if category_text:
#                     row_data[2] = category_text

#                 print(f"Extracted: {row_data}")
#             else:
#                 print(f"No valid link found for: {package_id}")

#         except Exception as e:
#             print(f"Error processing {package_id}: {e}")

#         dics.append(row_data)
#         sleep(2)

#     browser.quit()
#     return pd.DataFrame(dics, columns=['packageName', 'appName', 'category', 'source'])


# # Load the existing app_category_1.csv file
# df = pd.read_csv(os.path.join(PATH_DATA, 'app_category_1.csv'))

# # Extract additional categories for apps that are not yet categorized
# apps_to_categorize = df[df['category'].isna()]
# res = extractAppCategory2(apps_to_categorize)

# # Update original DataFrame with newly extracted information
# df.set_index('packageName', inplace=True)
# res.set_index('packageName', inplace=True)
# df.update(res)
# df.reset_index(inplace=True)

# # Save updated DataFrame to CSV
# print(df.head(), "\n", df.columns, "\n", df.shape, "\n")
# df.to_csv(os.path.join(PATH_DATA, "app_category_1.csv"), index=False)
# print(df.isna().sum())
import os
import pandas as pd
import json
from tqdm import tqdm
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.firefox.options import Options as FirefoxOptions
from selenium.webdriver.common.desired_capabilities import DesiredCapabilities
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
from time import sleep

# Part 3: Categorize the apps, second stage
print("<"*20 + " Second Extraction Began " + ">"*20)
sleep(1)

if os.path.exists(os.path.join(PATH_DATA, "app_category_2.csv")):
    os.remove(os.path.join(PATH_DATA, "app_category_2.csv"))

def extractAppCategory2(apps: pd.DataFrame) -> pd.DataFrame:
    dics = []

    # Set up browser options
    opts = FirefoxOptions()
    opts.add_argument("--headless")  # Remove this line for debugging in non-headless mode
    
    # Set up capabilities to force English content
    capabilities = DesiredCapabilities.FIREFOX
    capabilities["marionette"] = True
    capabilities["acceptInsecureCerts"] = True
    opts.set_preference("intl.accept_languages", "en-US, en")

    # Launch the browser with specified capabilities
    browser = webdriver.Firefox(options=opts, capabilities=capabilities)
    actions = ActionChains(browser)

    for idx, row in tqdm(apps.iterrows(), total=apps.shape[0]):
        package_id = row['packageName']
        if pd.notna(row['category']):
            continue  # Skip if the category is already available

        row_data = [package_id, None, None, "https://play.google.com"]
        # search_platform = f"https://www.google.com/search?q={package_id}+site:play.google.com"
        search_platform = f'https://play.google.com/store/apps/details?id={package_id}&hl=en'

        try:
            # Search for the app on Google
            browser.get(search_platform)
            sleep(3)  # Wait for search results to load

            # Extract the first Google Play link
            anchor_tags = browser.find_elements(By.TAG_NAME, "a")
            links = [tag.get_attribute("href") for tag in anchor_tags if tag.get_attribute("href") is not None]
            links = [link for link in links if "https://play.google.com/store/apps/details" in link]

            # Debugging: Print out the links being found
            print("Extracted Links:", links)

            # Correct the URL handling to ensure the package ID is retained
            if links:
                # Use the first valid link directly
                target_link = links[0]

                # Ensure it has the correct base URL and add `hl=en` if needed
                if "hl=" not in target_link:
                    if "?" in target_link:
                        clean_url = target_link + "&hl=en"
                    else:
                        clean_url = target_link + "?hl=en"
                else:
                    clean_url = target_link

                print("Navigating to Clean URL:", clean_url)  # Debugging print
                browser.get(clean_url)
                sleep(5)  # Wait for the Play Store page to load

                # Explicit wait for the span containing the app name
                try:
                    WebDriverWait(browser, 30).until(
                        EC.presence_of_element_located((By.XPATH, "//h1/span[@itemprop='name']"))
                    )
                except Exception as e:
                    print(f"Explicit wait failed for app name: {e}")

                # Scroll down to ensure dynamic content is fully loaded
                browser.execute_script("window.scrollTo(0, document.body.scrollHeight);")
                sleep(3)  # Wait for the page to fully load after scrolling

                # Debugging: Print the HTML source to verify element presence
                print("=== Page Source Start ===")
                print(browser.page_source)
                print("=== Page Source End ===")

                # Parse the page with BeautifulSoup
                soup = BeautifulSoup(browser.page_source, 'html.parser')

                # Extract app name using itemprop attribute
                app_name_tag = soup.find('span', {'itemprop': 'name'})
                if app_name_tag:
                    row_data[1] = app_name_tag.text.strip()
                    print(f"App Name Extracted: {row_data[1]}")
                else:
                    # Fallback methods to find the app name in case `itemprop="name"` does not exist
                    app_name_tag_alt = soup.find('h1')  # Try finding an <h1> tag as an alternative
                    if app_name_tag_alt:
                        row_data[1] = app_name_tag_alt.text.strip()
                        print(f"App Name Extracted (Fallback): {row_data[1]}")

                # Extract category using different possible methods
                category_text = None

                # Method 1: Search for the JSON-LD script with application/ld+json
                json_ld_script = soup.find('script', {'type': 'application/ld+json'})
                if json_ld_script:
                    try:
                        json_data = json.loads(json_ld_script.string)
                        if "applicationCategory" in json_data:
                            category_text = json_data["applicationCategory"].strip().upper()
                    except json.JSONDecodeError as e:
                        print(f"JSON parsing error: {e}")

                # Method 2: Search for a div with itemprop="genre"
                if not category_text:
                    genre_div = soup.find('div', {'itemprop': 'genre'})
                    if genre_div:
                        # Try extracting from <span> first
                        genre_span = genre_div.find('span', {'aria-hidden': 'true'})
                        if genre_span:
                            category_text = genre_span.text.strip().upper()
                        # Try extracting from <a> tag if <span> is not found
                        if not category_text:
                            genre_a = genre_div.find('a', {'aria-label': True})
                            if genre_a:
                                category_text = genre_a['aria-label'].strip().upper()

                # Method 3: Search for category within a known structure
                if not category_text:
                    category_a_tags = soup.find_all('a', href=True)
                    for a_tag in category_a_tags:
                        if "/store/apps/category/" in a_tag['href']:
                            category_text = a_tag.text.strip().upper()
                            break

                # Method 4: Search by potential keywords for a fallback approach
                if not category_text:
                    potential_spans = soup.find_all('span')
                    for span in potential_spans:
                        if 'Category' in span.text:
                            next_tag = span.find_next()
                            if next_tag:
                                category_text = next_tag.text.strip().upper()
                                break

                # Update the row_data if category_text is found
                if category_text:
                    row_data[2] = category_text

                print(f"Extracted: {row_data}")
            else:
                print(f"No valid link found for: {package_id}")

        except Exception as e:
            print(f"Error processing {package_id}: {e}")

        dics.append(row_data)
        sleep(2)

    browser.quit()
    return pd.DataFrame(dics, columns=['packageName', 'appName', 'category', 'source'])

# Load the existing app_category_1.csv file
df = pd.read_csv(os.path.join(PATH_DATA, 'app_category_1.csv'))

# Extract additional categories for apps that are not yet categorized
apps_to_categorize = df[df['category'].isna()]
res = extractAppCategory2(apps_to_categorize)

# Update original DataFrame with newly extracted information
df.set_index('packageName', inplace=True)
res.set_index('packageName', inplace=True)
df.update(res)
df.reset_index(inplace=True)

# Save updated DataFrame to CSV
print(df.head(), "\n", df.columns, "\n", df.shape, "\n")
df.to_csv(os.path.join(PATH_DATA, "app_category_1.csv"), index=False)
print(df.isna().sum())

In [ ]:
import os
import pandas as pd
from bs4 import BeautifulSoup
import requests
from tqdm import tqdm

# Function to extract app categories using additional online search
def extractAppCategory3(app: pd.DataFrame) -> pd.DataFrame:
    for idx, row in tqdm(app.iterrows(), total=app.shape[0]):
        package_id = row['packageName']
        
        # Skip if 'category' is already filled
        if pd.notna(row.get('category')):
            continue
        
        try:
            search_url = f'https://apkpure.net/search?q={package_id}'
            search_response = requests.get(search_url)
            search_soup = BeautifulSoup(search_response.text, 'html.parser')
            
            # Extract URL for the app details page
            url_tag = search_soup.find('a', class_='first-info')
            if url_tag:
                app_url = url_tag.get('href')
                if not app_url.startswith('http'):
                    app_url = 'https://apkpure.net' + app_url
                print(f"Extracted Link: {app_url}")  # Debugging print
                response = requests.get(app_url)
                soup = BeautifulSoup(response.text, 'html.parser')
                
                # Extract category information
                information_box = soup.find('div', class_='information-box')
                if not information_box:
                    print("Information Box not found.")
                    continue  # Skip to the next iteration

                apk_info = information_box.find('div', class_='apk-info')
                if not apk_info:
                    print("APK Info not found.")
                    continue

                row_div = apk_info.find('div', class_='row')
                if not row_div:
                    print("Row Div not found.")
                    continue

                # Find all 'div' elements with class 'info' inside the row_div
                info_divs = row_div.find_all('div', class_='info')
                category_found = False
                for info_div in info_divs:
                    # Find the 'div' with class 'title'
                    title_div = info_div.find('div', class_='title')
                    title_text = title_div.get_text(strip=True) if title_div else None
                    if title_text == 'Category':
                        # The category might be in 'a' or 'div' with class 'additional-info'
                        category_tag = info_div.find(class_='additional-info')
                        if category_tag and category_tag.get_text(strip=True):
                            category = category_tag.get_text(strip=True)
                            app.at[idx, 'category'] = category.upper()
                            print(f"Category Extracted: {category.upper()}")
                            category_found = True
                            break  # Exit after finding the category
                if not category_found:
                    print("Category not found.")

                # Extract app name information
                title_tag = soup.find('title')
                if title_tag:
                    app_name = title_tag.text.strip()
                    app.at[idx, 'appName'] = app_name
                    print(f"App Name Extracted: {app_name}")

                app.at[idx, 'source'] = 'apkpure.net'
            else:
                print(f"No app found for package ID: {package_id}")
        except Exception as e:
            print(f"Error processing {package_id}: {e}")
    return app



# Load the app_category_1.csv file
df = pd.read_csv(os.path.join(PATH_DATA, "app_category_1.csv"), index_col=False, header=0)
print(df.isna().sum())
print("-"*50)

# Extract additional categories and update the DataFrame
df = extractAppCategory3(df)
print(df.isna().sum())

# Drop unnecessary columns and save the updated CSV
df.drop(columns=[col for col in df.columns if 'Unnamed' in col], inplace=True)
df.to_csv(os.path.join(PATH_DATA, "app_category_1.csv"), index=False)


In [ ]:
apps = pd.read_csv(os.path.join(PATH_DATA, 'app_category_1.csv'), index_col=0)

for uid in tqdm(uids):
    if ".csv" in uid: continue
    df = pd.read_csv(os.path.join(PATH_SENSOR, uid, 'KeyEvent.csv'), index_col=0)
    package_category_mapping = dict(zip(apps.index, apps['category']))
    df['category'] = df['packageName'].map(package_category_mapping)
    df.to_csv(os.path.join(PATH_SENSOR, uid, 'KeyEvent.csv'))
